In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader
from source.version3.model import EfficientModel
from source.version3.score import ScoreDataset, scoreModel

In [3]:
def score(fold):
    data = pd.read_csv('../../data/raw/data.csv')
    data = data[data['fold'] == fold].reset_index(drop=True)
    driver = data[['image_id','label']].copy()
    loader = {}
    loader['path'] = '../../data/raw/train_images/'
    loader['data'] = data
    valid = ScoreDataset(**loader)
    valid = DataLoader(valid, batch_size=10, shuffle=False, num_workers=1, drop_last=False)
    model = EfficientModel()
    weights = torch.load('../../model/version3/model_{}.pt'.format(fold), map_location='cpu')
    weights = weights['model_state_dict']
    weights = {name.replace('module.model','model'): param for name, param in weights.items()}
    try:
        weights.pop('n_averaged')
    except: 
        pass
    model.load_state_dict(weights)
    model = model.to('cuda:0')
    score = scoreModel(model, valid)
    score = pd.DataFrame(score)
    score.columns = ['score_0','score_1','score_2','score_3','score_4']
    driver = driver.join(score) 
    driver.to_csv('../../model/version3/valid_{}.csv'.format(fold), index=False)
    model.cpu()
    del model
    return None

In [4]:
score(0)

In [5]:
score(1)

In [6]:
score(2)

In [7]:
score(3)

In [8]:
score(4)

In [9]:
data0 = pd.read_csv('../../model/version3/valid_0.csv')
data1 = pd.read_csv('../../model/version3/valid_1.csv')
data2 = pd.read_csv('../../model/version3/valid_2.csv')
data3 = pd.read_csv('../../model/version3/valid_3.csv')
data4 = pd.read_csv('../../model/version3/valid_4.csv')

In [10]:
data = data0.append(data1).append(data2).append(data3).append(data4)
data = data.groupby(['image_id','label']).mean().reset_index()

In [11]:
data.to_csv('../../score/version3.csv', index=False)

In [12]:
data.shape

(21397, 7)

In [13]:
data.head()

,image_id,label,score_0,score_1,score_2,score_3,score_4
0,1000015157.jpg,0,2.482042e-01,4.149383e-02,6.803436e-01,1.729214e-09,2.995834e-02
1,1000201771.jpg,3,8.824922e-34,7.309640e-28,2.616991e-29,1.000000e+00,2.938626e-37
2,100042118.jpg,1,2.553375e-08,2.043551e-08,6.140899e-09,1.123080e-08,1.000000e+00
3,1000723321.jpg,1,3.152156e-11,1.000000e+00,1.151280e-12,4.458672e-10,7.415820e-09
4,1000812911.jpg,3,2.578005e-19,6.496691e-21,2.360277e-17,1.000000e+00,7.645933e-19
